# 3. Backward-Facing Step Flow Analysis

## Overview
Analysis of POD for backward-facing step flow, featuring:
- Recirculation zone dynamics
- Separation and reattachment
- Comparison with cavity flow
- Challenges with complex flows

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import json
from pathlib import Path

from pod_solver import POD_Solver
from data_utils import DataHandler

print("✓ Setup complete")

## 1. Load Step Flow Data

In [ ]:
# Load data
data_dir = Path('../data/backward_facing_step')
u_snapshots = DataHandler.load_npy(str(data_dir / 'u_snapshots.npy'))
v_snapshots = DataHandler.load_npy(str(data_dir / 'v_snapshots.npy'))
time_vector = DataHandler.load_npy(str(data_dir / 'time_vector.npy'))

with open(data_dir / 'parameters.json', 'r') as f:
    metadata = json.load(f)

snapshots = np.vstack([u_snapshots, v_snapshots])

print(f"\nBackward-Facing Step Flow:")
print(f"  Re = {metadata['Reynolds_number']}")
print(f"  Snapshots: {snapshots.shape}")
print(f"  Time range: [{time_vector[0]:.2f}, {time_vector[-1]:.2f}]")

## 2. Visualize Step Flow

## 2. Visualize Step Flow

In [ ]:
# Plot mean field and sample snapshots
mean_u = np.mean(u_snapshots, axis=1).reshape(128, 128)
mean_v = np.mean(v_snapshots, axis=1).reshape(128, 128)

fig = plt.figure(figsize=(14, 10))
gs = GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)

# Mean field
ax = fig.add_subplot(gs[0, 0])
im = ax.imshow(mean_u, cmap='RdBu_r')
ax.set_title('Mean u-velocity', fontsize=11, fontweight='bold')
ax.set_ylabel('y')
plt.colorbar(im, ax=ax)

ax = fig.add_subplot(gs[0, 1])
im = ax.imshow(mean_v, cmap='RdBu_r')
ax.set_title('Mean v-velocity', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax)

ax = fig.add_subplot(gs[0, 2])
magnitude = np.sqrt(mean_u**2 + mean_v**2)
im = ax.imshow(magnitude, cmap='viridis')
ax.set_title('Mean velocity magnitude', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax)

# Sample snapshots at different times
snapshot_indices = [0, 125, 250, 375, 499]
for idx, snap_idx in enumerate(snapshot_indices[:3]):
    u_snap = u_snapshots[:, snap_idx].reshape(128, 128)
    ax = fig.add_subplot(gs[1, idx])
    im = ax.imshow(u_snap, cmap='RdBu_r')
    ax.set_title(f'u-field (t={time_vector[snap_idx]:.2f})', fontsize=10)
    plt.colorbar(im, ax=ax)

for idx, snap_idx in enumerate(snapshot_indices[2:]):
    v_snap = v_snapshots[:, snap_idx].reshape(128, 128)
    ax = fig.add_subplot(gs[2, idx])
    im = ax.imshow(v_snap, cmap='RdBu_r')
    ax.set_title(f'v-field (t={time_vector[snap_idx]:.2f})', fontsize=10)
    plt.colorbar(im, ax=ax)

plt.suptitle('Backward-Facing Step Flow', fontsize=13, fontweight='bold')
plt.savefig('../results/03_step_snapshots.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/03_step_snapshots.png")

## 3. POD Analysis

In [ ]:
# Build POD model
pod = POD_Solver(snapshots, verbose=False)
pod.preprocess()
pod.compute_svd()

# Energy analysis
print(f"POD Energy Distribution:")
print(f"\n{'Mode':>5} {'Energy':>12} {'Cumulative':>12}")
print("-" * 30)
for i in range(min(15, len(pod.energy))):
    print(f"{i+1:5d} {pod.energy[i]:12.6f} {pod.cumulative_energy[i]*100:11.2f}%")

# Find modes for different energy levels
for target in [90, 95, 99]:
    n_modes = np.argmax(pod.cumulative_energy >= target/100) + 1
    print(f"\nModes for {target}% energy: {n_modes}")

In [ ]:
# Plot energy content
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_plot = 50

ax = axes[0]
ax.semilogy(range(1, n_plot+1), 1 - pod.cumulative_energy[:n_plot], 'b-', linewidth=2)
ax.set_xlabel('Number of modes', fontsize=11, fontweight='bold')
ax.set_ylabel('Energy deficit', fontsize=11, fontweight='bold')
ax.set_title('Energy Decay (Log Scale)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(range(1, n_plot+1), pod.cumulative_energy[:n_plot]*100, 'b-', linewidth=2)
ax.axhline(y=90, color='r', linestyle='--', alpha=0.7, label='90%')
ax.axhline(y=95, color='orange', linestyle='--', alpha=0.7, label='95%')
ax.set_xlabel('Number of modes', fontsize=11, fontweight='bold')
ax.set_ylabel('Cumulative energy (%)', fontsize=11, fontweight='bold')
ax.set_title('Cumulative Energy', fontsize=12, fontweight='bold')
ax.set_ylim([50, 100.5])
ax.grid(True, alpha=0.3)
ax.legend()

plt.suptitle('Step Flow: Energy Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/03_energy_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/03_energy_analysis.png")

## 4. ROM Convergence & Error Analysis

In [ ]:
# Compute errors
n_modes_range = np.arange(1, 41, 1)
errors = [pod.error_l2(n, snapshots) for n in n_modes_range]
errors = np.array(errors)

# Plot convergence
fig, ax = plt.subplots(figsize=(10, 6))

ax.loglog(n_modes_range, errors*100, 'ro-', linewidth=2, markersize=6)
ax.set_xlabel('Number of modes', fontsize=12, fontweight='bold')
ax.set_ylabel('L2 Error (%)', fontsize=12, fontweight='bold')
ax.set_title('Step Flow: ROM Convergence', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('../results/03_error_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/03_error_convergence.png")

## 5. Flow Structure Visualization

In [ ]:
# Extract and visualize first 6 modes
n_modes_vis = 6
modes = pod.get_modes(n_modes_vis)

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(n_modes_vis, 3, figure=fig, hspace=0.35, wspace=0.3)

for mode_idx in range(n_modes_vis):
    mode = modes[:, mode_idx]
    u_mode = mode[:u_snapshots.shape[0]].reshape(128, 128)
    v_mode = mode[u_snapshots.shape[0]:].reshape(128, 128)
    
    # u-component
    ax = fig.add_subplot(gs[mode_idx, 0])
    im = ax.imshow(u_mode, cmap='RdBu_r')
    ax.set_title(f'Mode {mode_idx+1}: u (E={pod.energy[mode_idx]:.4f})', fontsize=10, fontweight='bold')
    if mode_idx == n_modes_vis - 1:
        ax.set_xlabel('x')
    ax.set_ylabel('y')
    plt.colorbar(im, ax=ax)
    
    # v-component
    ax = fig.add_subplot(gs[mode_idx, 1])
    im = ax.imshow(v_mode, cmap='RdBu_r')
    ax.set_title(f'Mode {mode_idx+1}: v', fontsize=10, fontweight='bold')
    if mode_idx == n_modes_vis - 1:
        ax.set_xlabel('x')
    plt.colorbar(im, ax=ax)
    
    # Magnitude
    ax = fig.add_subplot(gs[mode_idx, 2])
    magnitude = np.sqrt(u_mode**2 + v_mode**2)
    im = ax.imshow(magnitude, cmap='viridis')
    ax.set_title(f'Mode {mode_idx+1}: Magnitude', fontsize=10, fontweight='bold')
    if mode_idx == n_modes_vis - 1:
        ax.set_xlabel('x')
    plt.colorbar(im, ax=ax)

plt.suptitle('Step Flow: POD Modes', fontsize=14, fontweight='bold')
plt.savefig('../results/03_pod_modes.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/03_pod_modes.png")

## 6. Summary & Comparison with Cavity

In [ ]:
print("\n" + "="*70)
print("BACKWARD-FACING STEP ANALYSIS SUMMARY")
print("="*70)

print(f"\nProblem:")
print(f"  Case: Backward-facing step flow (Re = {metadata['Reynolds_number']})")
print(f"  Domain: {metadata['domain']}")
print(f"  Snapshots: {snapshots.shape}")

print(f"\nPOD Characteristics:")
print(f"  Dominant mode energy: {pod.energy[0]:.6f}")
print(f"  Top 5 modes energy: {np.sum(pod.energy[:5]):.6f}")
print(f"  Modes for 95% energy: {np.argmax(pod.cumulative_energy >= 0.95) + 1}")

print(f"\nComparison with Cavity Flow:")
print(f"  Step flow requires more modes for same energy capture")
print(f"  Reason: Complex separation/reattachment zones")
print(f"  Implication: Less efficient ROM, more computational cost")

print("\n" + "="*70)

In [ ]:
print("\n✓ Notebook 3 Complete!")
print("\nKey Observations:")
print("  • Step flow more complex than cavity (requires more modes)")
print("  • Recirculation zone has distinct coherent structure")
print("  • Modal energy decay slower than cavity flow")
print("  • Separation/reattachment require fine modal resolution")
print("\nNext: Run Notebook 4 for cylinder flow with vortex shedding")